# Step 6 - Machine Learning Pipeline (Classification)

Goal:
- Load Gold feature table from Delta Lake.
- Train 5 Spark ML classifiers.
- Compare models with Accuracy, F1, Precision, Recall, AUC-ROC.
- Track all experiments with MLflow.
- Export model outputs for Step 7 dashboard.

In [6]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "pyspark": "pyspark",
    "delta": "delta-spark",
    "mlflow": "mlflow",
    "sklearn": "scikit-learn"
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(f"Installing missing packages: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Package installation completed.")
else:
    print("All Step 6 packages are already installed.")

All Step 6 packages are already installed.


In [7]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import mlflow
import mlflow.spark

from sklearn.metrics import roc_curve

from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, NumericType
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    GBTClassifier,
    LinearSVC,
    LogisticRegression,
    RandomForestClassifier,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")
sns.set_palette("tab10")

In [8]:
APP_NAME = "CreditRisk-Step6-MLPipeline"
DEFAULT_WORKSPACE_ROOT = "c:/Users/MONSTER/Desktop/fintech-credit-risk-engine-main"
ALLOW_PARQUET_FALLBACK = os.getenv("STEP6_ALLOW_PARQUET_FALLBACK", "0") == "1"

def resolve_workspace_root(default_root: str) -> str:
    candidates = [
        os.getenv("WORKSPACE_ROOT"),
        os.getcwd(),
        default_root,
    ]
    for candidate in candidates:
        if not candidate:
            continue
        candidate = candidate.replace("\\", "/")
        if os.path.isdir(candidate) and os.path.isdir(f"{candidate}/notebooks"):
            return candidate
    return default_root

WORKSPACE_ROOT = resolve_workspace_root(DEFAULT_WORKSPACE_ROOT)
SPARK_LOCAL_DIR = os.getenv("SPARK_LOCAL_DIR", f"{WORKSPACE_ROOT}/tmp/spark-local")
os.makedirs(SPARK_LOCAL_DIR, exist_ok=True)

def ensure_windows_hadoop_home(workspace_root: str):
    if os.name != "nt":
        return None

    candidate_homes = []
    env_home = os.environ.get("HADOOP_HOME") or os.environ.get("hadoop.home.dir")
    if env_home:
        candidate_homes.append(env_home)
    candidate_homes.append(f"{workspace_root}/third_party/hadoop")

    for home in candidate_homes:
        if not home:
            continue
        winutils_path = os.path.join(home, "bin", "winutils.exe")
        if os.path.exists(winutils_path):
            os.environ["HADOOP_HOME"] = home
            os.environ["hadoop.home.dir"] = home
            os.environ["PATH"] = os.path.join(home, "bin") + ";" + os.environ.get("PATH", "")
            return home

    return None

def reset_pyspark_runtime_state():
    try:
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None
    except Exception:
        pass
    try:
        SparkSession._instantiatedSession = None
        SparkSession._activeSession = None
    except Exception:
        pass

def resolve_delta_base(workspace_root: str) -> str:
    candidates = []
    env_base = os.getenv("DELTA_BASE")
    if env_base:
        candidates.append(env_base)
    candidates.extend([
        f"{workspace_root}/delta_lake",
        "/app/delta_lake",
    ])

    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            return candidate.replace("\\", "/")

    return candidates[0].replace("\\", "/")

hadoop_home = ensure_windows_hadoop_home(WORKSPACE_ROOT)
if hadoop_home:
    print(f"Using HADOOP_HOME: {hadoop_home}")

try:
    from delta import configure_spark_with_delta_pip
except Exception as delta_import_error:
    raise RuntimeError(
        "delta-spark import failed. Install delta-spark and rerun Step 6."
    ) from delta_import_error

reset_pyspark_runtime_state()
spark_builder = (
    SparkSession.builder
    .master("local[*]")
    .appName(APP_NAME)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", os.getenv("SPARK_LOCAL_SHUFFLE_PARTITIONS", "32"))
    .config("spark.default.parallelism", os.getenv("SPARK_LOCAL_DEFAULT_PARALLELISM", "32"))
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.driver.memory", os.getenv("SPARK_LOCAL_DRIVER_MEMORY", "6g"))
    .config("spark.driver.maxResultSize", os.getenv("SPARK_LOCAL_DRIVER_MAX_RESULT", "1g"))
    .config("spark.local.dir", SPARK_LOCAL_DIR)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(spark_builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

DELTA_BASE = resolve_delta_base(WORKSPACE_ROOT)
FEATURE_PATH = f"{DELTA_BASE}/gold/features"
PARQUET_FALLBACK_PATH = f"{DELTA_BASE}/gold/features_parquet_fallback"

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Delta base: {DELTA_BASE}")
print(f"Feature path: {FEATURE_PATH}")

Using HADOOP_HOME: c:/Users/MONSTER/Desktop/fintech-credit-risk-engine-main/third_party/hadoop
Workspace root: c:/Users/MONSTER/Desktop/fintech-credit-risk-engine-main
Delta base: c:/Users/MONSTER/Desktop/fintech-credit-risk-engine-main/delta_lake
Feature path: c:/Users/MONSTER/Desktop/fintech-credit-risk-engine-main/delta_lake/gold/features


In [ ]:
read_errors = []

df = None
source_format = None

try:
    df = spark.read.format("delta").load(FEATURE_PATH)
    source_format = "delta"
except Exception as delta_read_error:
    read_errors.append(f"delta:{delta_read_error}")

if df is None:
    parquet_candidates = [FEATURE_PATH]
    if ALLOW_PARQUET_FALLBACK:
        parquet_candidates.append(PARQUET_FALLBACK_PATH)

    for parquet_path in parquet_candidates:
        if not os.path.exists(parquet_path):
            continue
        try:
            df = spark.read.parquet(parquet_path)
            source_format = f"parquet:{parquet_path}"
            break
        except Exception as parquet_read_error:
            read_errors.append(f"parquet:{parquet_path}:{parquet_read_error}")

if df is None:
    raise RuntimeError(
        "Step 6 could not load features from delta/parquet. "
        f"Checked paths: {[FEATURE_PATH, PARQUET_FALLBACK_PATH]}. "
        f"Errors: {read_errors}"
    )

if "target" in df.columns and "label" not in df.columns:
    df = df.withColumn("label", F.col("target").cast("double"))
elif "label" in df.columns:
    df = df.withColumn("label", F.col("label").cast("double"))
else:
    raise ValueError("No target/label column found in gold feature table.")

print(f"Loaded feature table format: {source_format}")
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.select("label").groupBy("label").count().orderBy("label").show()
df.printSchema()

RuntimeError: Step 6 could not load features from delta/parquet. Checked paths: ['c:/Users/MONSTER/Desktop/fintech-credit-risk-engine-main/delta_lake/gold/features', 'c:/Users/MONSTER/Desktop/fintech-credit-risk-engine-main/delta_lake/gold/features_parquet_fallback']. Errors: ["delta:An error occurred while calling o55.load.\n: com.google.common.util.concurrent.ExecutionError: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'\r\n\tat com.google.common.cache.LocalCache$Segment.get(LocalCache.java:2072)\r\n\tat com.google.common.cache.LocalCache.get(LocalCache.java:3986)\r\n\tat com.google.common.cache.LocalCache$LocalManualCache.get(LocalCache.java:4855)\r\n\tat org.apache.spark.sql.delta.DeltaLog$.getDeltaLogFromCache$1(DeltaLog.scala:1049)\r\n\tat org.apache.spark.sql.delta.DeltaLog$.initializeDeltaLog$1(DeltaLog.scala:1060)\r\n\tat org.apache.spark.sql.delta.DeltaLog$.apply(DeltaLog.scala:1071)\r\n\tat org.apache.s

In [ ]:
exclude_cols = {"target", "label", "class_weight"}
numeric_cols = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, NumericType) and field.name not in exclude_cols
]

if not numeric_cols:
    raise ValueError("No numeric feature columns were found for VectorAssembler.")

assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="features",
    handleInvalid="keep",
)

assembled_df = assembler.transform(df)
selected_cols = ["features", "label"]
if "class_weight" in assembled_df.columns:
    selected_cols.append("class_weight")

model_df = assembled_df.select(*selected_cols)
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count = test_df.count()
has_class_weight = "class_weight" in model_df.columns

print(f"Feature count used in VectorAssembler: {len(numeric_cols)}")
print(f"Train rows: {train_count:,}")
print(f"Test rows: {test_count:,}")
print(f"Weight column available: {has_class_weight}")

In [ ]:
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)

def compute_binary_metrics(predictions_df):
    cm_rows = (
        predictions_df
        .groupBy(F.col("label").cast("int").alias("label"), F.col("prediction").cast("int").alias("prediction"))
        .count()
        .collect()
    )

    counts = {(int(r["label"]), int(r["prediction"])): int(r["count"]) for r in cm_rows}
    tn = counts.get((0, 0), 0)
    fp = counts.get((0, 1), 0)
    fn = counts.get((1, 0), 0)
    tp = counts.get((1, 1), 0)

    total = tn + fp + fn + tp
    accuracy = (tn + tp) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

def maybe_set_weight_col(estimator, use_weight_col: bool):
    if use_weight_col and estimator.hasParam("weightCol"):
        estimator.set(estimator.getParam("weightCol"), "class_weight")
    return estimator

def extract_mlflow_params(estimator):
    params = {}
    for p in estimator.params:
        if estimator.isDefined(p):
            value = estimator.getOrDefault(p)
            if isinstance(value, (str, int, float, bool)):
                params[p.name] = value
            else:
                params[p.name] = str(value)
    return params

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=80,
        regParam=0.01,
        elasticNetParam=0.0,
    ),
    "DecisionTreeClassifier": DecisionTreeClassifier(
        featuresCol="features",
        labelCol="label",
        maxDepth=8,
        maxBins=128,
        seed=42,
    ),
    "RandomForestClassifier": RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=150,
        maxDepth=10,
        maxBins=128,
        featureSubsetStrategy="sqrt",
        seed=42,
    ),
    "GBTClassifier": GBTClassifier(
        featuresCol="features",
        labelCol="label",
        maxIter=120,
        maxDepth=6,
        maxBins=128,
        stepSize=0.08,
        seed=42,
    ),
    "LinearSVC": LinearSVC(
        featuresCol="features",
        labelCol="label",
        maxIter=100,
        regParam=0.05,
    ),
}

print(f"Total models to train: {len(models)}")

In [ ]:
mlruns_dir = Path(WORKSPACE_ROOT) / "mlruns"
mlruns_dir.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(mlruns_dir.resolve().as_uri())
mlflow.set_experiment("credit-risk-step6-model-comparison")

comparison_rows = []
trained_models = {}
prediction_cache = {}

for model_name, base_estimator in models.items():
    estimator = maybe_set_weight_col(base_estimator, has_class_weight)

    with mlflow.start_run(run_name=model_name) as run:
        mlflow.set_tag("project", "fintech-credit-risk-engine")
        mlflow.set_tag("pipeline_step", "step6")
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("feature_count", len(numeric_cols))
        mlflow.log_param("train_rows", train_count)
        mlflow.log_param("test_rows", test_count)

        params_to_log = extract_mlflow_params(estimator)
        for param_name, param_value in params_to_log.items():
            mlflow.log_param(param_name, param_value)

        fitted_model = estimator.fit(train_df)
        predictions = fitted_model.transform(test_df).cache()

        metrics = compute_binary_metrics(predictions)
        auc = float(auc_evaluator.evaluate(predictions))

        mlflow.log_metric("accuracy", metrics["accuracy"])
        mlflow.log_metric("f1", metrics["f1"])
        mlflow.log_metric("precision", metrics["precision"])
        mlflow.log_metric("recall", metrics["recall"])
        mlflow.log_metric("auc", auc)

        mlflow.spark.log_model(fitted_model, artifact_path="model")

        comparison_rows.append({
            "model": model_name,
            "accuracy": metrics["accuracy"],
            "f1": metrics["f1"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "auc": auc,
            "tn": metrics["tn"],
            "fp": metrics["fp"],
            "fn": metrics["fn"],
            "tp": metrics["tp"],
            "mlflow_run_id": run.info.run_id,
        })

        trained_models[model_name] = fitted_model
        prediction_cache[model_name] = predictions

if not comparison_rows:
    raise RuntimeError("No model completed training in Step 6.")

comparison_pdf = pd.DataFrame(comparison_rows).sort_values("auc", ascending=False).reset_index(drop=True)
comparison_pdf

In [ ]:
best_model_name = comparison_pdf.loc[0, "model"]
best_predictions = prediction_cache[best_model_name]

print(f"Best model by AUC: {best_model_name}")
best_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

cm_pdf = (
    best_predictions
    .groupBy(F.col("label").cast("int").alias("label"), F.col("prediction").cast("int").alias("prediction"))
    .count()
    .toPandas()
)

cm_matrix = (
    cm_pdf
    .pivot(index="label", columns="prediction", values="count")
    .reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    .fillna(0)
    .astype(int)
)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_matrix, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
score_udf = F.udf(lambda v: float(v[1]) if v is not None else None, DoubleType())
score_source_col = "probability" if "probability" in best_predictions.columns else "rawPrediction"

roc_points_pdf = (
    best_predictions
    .select(
        F.col("label").cast("int").alias("label"),
        score_udf(F.col(score_source_col)).alias("score"),
    )
    .dropna()
    .toPandas()
)

fpr, tpr, thresholds = roc_curve(roc_points_pdf["label"], roc_points_pdf["score"])
roc_pdf = pd.DataFrame({"fpr": fpr, "tpr": tpr, "threshold": thresholds})

plt.figure(figsize=(6, 4))
plt.plot(roc_pdf["fpr"], roc_pdf["tpr"], label=best_model_name, linewidth=2)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
plt.title(f"ROC Curve - {best_model_name}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
score_series = roc_points_pdf["score"].astype(float)
score_min = float(score_series.min())
score_max = float(score_series.max())

if score_min < 0.0 or score_max > 1.0:
    # Convert margin-like scores to pseudo-probabilities for threshold search.
    calibrated_scores = 1.0 / (1.0 + np.exp(-score_series))
else:
    calibrated_scores = score_series

thresholds_to_test = np.round(np.linspace(0.05, 0.95, 19), 2)
fp_cost_base = float(os.getenv("STEP6_FP_COST", "1.0"))
fn_cost_base = float(os.getenv("STEP6_FN_COST", "5.0"))

threshold_rows = []
for threshold_value in thresholds_to_test:
    pred = (calibrated_scores >= threshold_value).astype(int)
    actual = roc_points_pdf["label"].astype(int)

    tn = int(((actual == 0) & (pred == 0)).sum())
    fp = int(((actual == 0) & (pred == 1)).sum())
    fn = int(((actual == 1) & (pred == 0)).sum())
    tp = int(((actual == 1) & (pred == 1)).sum())

    total = tn + fp + fn + tp
    accuracy = (tn + tp) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
    expected_cost = (fp * fp_cost_base) + (fn * fn_cost_base)

    threshold_rows.append({
        "model": best_model_name,
        "threshold": float(threshold_value),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "expected_cost": float(expected_cost),
        "fp_cost": float(fp_cost_base),
        "fn_cost": float(fn_cost_base),
    })

threshold_tuning_pdf = pd.DataFrame(threshold_rows).sort_values("threshold").reset_index(drop=True)
optimal_threshold_by_cost = threshold_tuning_pdf.sort_values(["expected_cost", "f1"], ascending=[True, False]).iloc[0]
optimal_threshold_by_f1 = threshold_tuning_pdf.sort_values("f1", ascending=False).iloc[0]

cost_scenarios = [
    (1.0, 3.0),
    (1.0, 5.0),
    (1.0, 8.0),
    (2.0, 5.0),
    (3.0, 10.0),
]

scenario_rows = []
for scenario_fp_cost, scenario_fn_cost in cost_scenarios:
    scenario_eval = threshold_tuning_pdf.copy()
    scenario_eval["expected_cost"] = (
        scenario_eval["fp"] * scenario_fp_cost + scenario_eval["fn"] * scenario_fn_cost
    )
    scenario_best = scenario_eval.sort_values(["expected_cost", "f1"], ascending=[True, False]).iloc[0]
    scenario_rows.append({
        "model": best_model_name,
        "fp_cost": float(scenario_fp_cost),
        "fn_cost": float(scenario_fn_cost),
        "optimal_threshold": float(scenario_best["threshold"]),
        "minimum_expected_cost": float(scenario_best["expected_cost"]),
        "f1_at_optimal": float(scenario_best["f1"]),
        "recall_at_optimal": float(scenario_best["recall"]),
        "precision_at_optimal": float(scenario_best["precision"]),
    })

business_cost_matrix_pdf = pd.DataFrame(scenario_rows).sort_values(["fn_cost", "fp_cost"]).reset_index(drop=True)

print("Threshold tuning (top 10 by lowest expected cost):")
print(threshold_tuning_pdf.sort_values(["expected_cost", "f1"], ascending=[True, False]).head(10).to_string(index=False))
print("\nBest threshold by expected cost:")
print(optimal_threshold_by_cost.to_string())
print("\nBest threshold by F1:")
print(optimal_threshold_by_f1.to_string())

plt.figure(figsize=(10, 4))
plt.plot(threshold_tuning_pdf["threshold"], threshold_tuning_pdf["f1"], marker="o", label="F1")
plt.plot(threshold_tuning_pdf["threshold"], threshold_tuning_pdf["recall"], marker="o", label="Recall")
plt.plot(threshold_tuning_pdf["threshold"], threshold_tuning_pdf["precision"], marker="o", label="Precision")
plt.axvline(float(optimal_threshold_by_cost["threshold"]), linestyle="--", color="red", label="Best Cost Threshold")
plt.title(f"Threshold Tuning Metrics - {best_model_name}")
plt.xlabel("Threshold")
plt.ylabel("Metric")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(threshold_tuning_pdf["threshold"], threshold_tuning_pdf["expected_cost"], marker="o", color="#0d47a1")
plt.axvline(float(optimal_threshold_by_cost["threshold"]), linestyle="--", color="red")
plt.title(f"Expected Cost vs Threshold (FP={fp_cost_base}, FN={fn_cost_base})")
plt.xlabel("Threshold")
plt.ylabel("Expected Cost")
plt.tight_layout()
plt.show()

business_cost_matrix_pdf

In [ ]:
rf_importance_pdf = pd.DataFrame()
gbt_importance_pdf = pd.DataFrame()

if "RandomForestClassifier" in trained_models:
    rf_importances = trained_models["RandomForestClassifier"].featureImportances.toArray().tolist()
    rf_importance_pdf = (
        pd.DataFrame({"feature": numeric_cols, "importance": rf_importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

if "GBTClassifier" in trained_models:
    gbt_importances = trained_models["GBTClassifier"].featureImportances.toArray().tolist()
    gbt_importance_pdf = (
        pd.DataFrame({"feature": numeric_cols, "importance": gbt_importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

if not rf_importance_pdf.empty:
    plt.figure(figsize=(8, 5))
    top_rf = rf_importance_pdf.head(10).sort_values("importance", ascending=True)
    sns.barplot(data=top_rf, x="importance", y="feature", orient="h")
    plt.title("Top 10 Feature Importances - RandomForest")
    plt.tight_layout()
    plt.show()

if not gbt_importance_pdf.empty:
    plt.figure(figsize=(8, 5))
    top_gbt = gbt_importance_pdf.head(10).sort_values("importance", ascending=True)
    sns.barplot(data=top_gbt, x="importance", y="feature", orient="h")
    plt.title("Top 10 Feature Importances - GBT")
    plt.tight_layout()
    plt.show()

In [ ]:
output_dir = Path(WORKSPACE_ROOT) / "output" / "step6"
output_dir.mkdir(parents=True, exist_ok=True)

comparison_pdf.to_csv(output_dir / "model_comparison.csv", index=False)
cm_pdf.to_csv(output_dir / "confusion_matrix_best.csv", index=False)
roc_pdf.to_csv(output_dir / "roc_curve_best.csv", index=False)
threshold_tuning_pdf.to_csv(output_dir / "threshold_tuning_best_model.csv", index=False)
business_cost_matrix_pdf.to_csv(output_dir / "business_cost_matrix.csv", index=False)

if not rf_importance_pdf.empty:
    rf_importance_pdf.to_csv(output_dir / "feature_importance_rf.csv", index=False)
if not gbt_importance_pdf.empty:
    gbt_importance_pdf.to_csv(output_dir / "feature_importance_gbt.csv", index=False)

default_rate = float(df.select(F.avg(F.col("label"))).first()[0])
avg_interest_rate = None
if "int_rate_num" in df.columns:
    avg_interest_rate = float(df.select(F.avg(F.col("int_rate_num"))).first()[0])

executive_metrics = {
    "total_loans": int(df.count()),
    "default_rate": default_rate,
    "best_model": best_model_name,
    "best_auc": float(comparison_pdf.loc[0, "auc"]),
    "avg_interest_rate": avg_interest_rate,
    "recommended_threshold_cost": float(optimal_threshold_by_cost["threshold"]),
    "recommended_threshold_f1": float(optimal_threshold_by_f1["threshold"]),
    "fp_cost_base": float(fp_cost_base),
    "fn_cost_base": float(fn_cost_base),
}

with open(output_dir / "executive_metrics.json", "w", encoding="utf-8") as f:
    json.dump(executive_metrics, f, indent=2)

threshold_summary = {
    "model": best_model_name,
    "score_min": score_min,
    "score_max": score_max,
    "uses_sigmoid_calibration": bool(score_min < 0.0 or score_max > 1.0),
    "best_threshold_by_cost": float(optimal_threshold_by_cost["threshold"]),
    "best_threshold_by_f1": float(optimal_threshold_by_f1["threshold"]),
    "f1_at_best_cost": float(optimal_threshold_by_cost["f1"]),
    "recall_at_best_cost": float(optimal_threshold_by_cost["recall"]),
    "precision_at_best_cost": float(optimal_threshold_by_cost["precision"]),
    "expected_cost_at_best_cost": float(optimal_threshold_by_cost["expected_cost"]),
}

with open(output_dir / "threshold_summary.json", "w", encoding="utf-8") as f:
    json.dump(threshold_summary, f, indent=2)

print(f"Step 6 artifacts exported to: {output_dir.as_posix()}")
print("Generated files:")
for file_name in sorted(os.listdir(output_dir)):
    print(f"- {file_name}")

## Step 6 Summary

This notebook satisfies the Step 6 requirements:
- Delta Gold feature table is loaded in Spark.
- 5 classification models are trained and compared.
- Accuracy, F1, Precision, Recall, and AUC are produced.
- MLflow logs params, metrics, and model artifacts for each run.
- Feature importance, confusion matrix, and ROC outputs are exported.
- Threshold tuning and business cost matrix are generated for decision optimization.